# 01 · A/B experiment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmavigo/hillstrom-uplift-lab/blob/main/notebooks/01_ab_experiment.ipynb)

**Business question:** did either email increase purchase conversion versus sending no email? Purchase conversion is the primary outcome. Visits and spend are secondary outcomes. The treatment is an email, not a discount.

## Run this notebook in Colab

Each notebook runs independently. The next cell installs the analysis packages, clones the project, and downloads the public dataset. You do not need Kaggle credentials.

In [ ]:
%pip -q install pandas numpy scipy statsmodels scikit-learn plotly


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path("/content/hillstrom-uplift-lab")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ericmavigo/hillstrom-uplift-lab.git", str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "download_data.py"], check=True)
print(f"Project directory: {REPO_DIR}")


## 1. Compare outcomes by randomized arm

We show absolute rates and the difference from the no-email control. A positive conversion difference means more customers purchased in the email group.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

df = pd.read_csv(REPO_DIR / "data" / "raw" / "hillstrom.csv")
control_name = "No E-Mail"
arms = ["Mens E-Mail", "Womens E-Mail"]
outcomes = {"visit": "Visit", "conversion": "Purchase", "spend": "Spend per customer"}


## 2. Estimate differences and uncertainty

For binary outcomes, use a two-proportion z-test and Newcombe-Wilson interval. For spend, use Welch's t-test and a bootstrap interval because spending is skewed and many customers spend zero. The primary conversion p-values are adjusted with Holm's method across the two campaign comparisons.

In [ ]:
def newcombe_difference_interval(y_t, y_c):
    p_t, p_c = y_t.mean(), y_c.mean()
    lo_t, hi_t = proportion_confint(int(y_t.sum()), len(y_t), method="wilson")
    lo_c, hi_c = proportion_confint(int(y_c.sum()), len(y_c), method="wilson")
    diff = p_t - p_c
    low = diff - np.sqrt((p_t - lo_t) ** 2 + (hi_c - p_c) ** 2)
    high = diff + np.sqrt((hi_t - p_t) ** 2 + (p_c - lo_c) ** 2)
    return low, high

def bootstrap_mean_difference(y_t, y_c, seed=42, reps=2500):
    rng = np.random.default_rng(seed)
    boot = np.empty(reps)
    for start in range(0, reps, 64):
        stop = min(start + 64, reps)
        mt = rng.choice(y_t, size=(stop-start, len(y_t)), replace=True).mean(axis=1)
        mc = rng.choice(y_c, size=(stop-start, len(y_c)), replace=True).mean(axis=1)
        boot[start:stop] = mt - mc
    return np.quantile(boot, [0.025, 0.975])

rows = []
for arm in arms:
    for outcome, label in outcomes.items():
        y_t = df.loc[df.segment == arm, outcome].to_numpy()
        y_c = df.loc[df.segment == control_name, outcome].to_numpy()
        difference = y_t.mean() - y_c.mean()
        if outcome in ("visit", "conversion"):
            p_value = proportions_ztest([y_t.sum(), y_c.sum()], [len(y_t), len(y_c)])[1]
            ci_low, ci_high = newcombe_difference_interval(y_t, y_c)
        else:
            p_value = stats.ttest_ind(y_t, y_c, equal_var=False).pvalue
            ci_low, ci_high = bootstrap_mean_difference(y_t, y_c)
        rows.append({"campaign": arm, "outcome": label, "n_treatment": len(y_t),
                     "treatment_mean": y_t.mean(), "control_mean": y_c.mean(),
                     "difference": difference, "ci_low": ci_low, "ci_high": ci_high,
                     "p_value": p_value})

results = pd.DataFrame(rows)
primary = results.outcome == "Purchase"
results["holm_adjusted_p"] = np.nan
results.loc[primary, "holm_adjusted_p"] = multipletests(results.loc[primary, "p_value"], method="holm")[1]
display(results.style.format({"treatment_mean": "{:.2%}", "control_mean": "{:.2%}",
                              "difference": "{:+.2%}", "ci_low": "{:+.2%}", "ci_high": "{:+.2%}",
                              "p_value": "{:.3g}", "holm_adjusted_p": "{:.3g}"}))


## 3. Visualize the primary outcome

The dashed line is the no-email control rate. Error bars show a confidence interval for the treatment-minus-control difference translated onto the treatment rate.

In [ ]:
import plotly.graph_objects as go

primary_results = results[results.outcome == "Purchase"]
fig = go.Figure()
for _, row in primary_results.iterrows():
    fig.add_trace(go.Bar(name=row.campaign, x=[row.campaign], y=[row.treatment_mean],
        error_y={"type": "data", "symmetric": False,
                 "array": [max(0, row.control_mean + row.ci_high - row.treatment_mean)],
                 "arrayminus": [max(0, row.treatment_mean - row.control_mean - row.ci_low)]}))
control_rate = df.loc[df.segment == control_name, "conversion"].mean()
fig.add_hline(y=control_rate, line_dash="dash", annotation_text=f"Control: {control_rate:.2%}")
fig.update_layout(title="Purchase conversion by email treatment", yaxis_title="Purchase conversion",
                  yaxis_tickformat=".1%", showlegend=False)
fig.show()


## 4. State the decision carefully

A campaign is evidence-supported on the primary metric when its Holm-adjusted p-value is below 0.05 and the estimated lift is positive. Statistical evidence does not establish profitability: the data do not contain message costs, product margins, or discount amounts. Use visits and spend as supporting outcomes, not as replacements chosen after looking at the results.

In [ ]:
for _, row in primary_results.iterrows():
    supported = row.difference > 0 and row.holm_adjusted_p < 0.05
    print(f"{row.campaign}: {'positive conversion evidence' if supported else 'no clear positive conversion evidence'}; "
          f"lift={row.difference:+.2%}, Holm p={row.holm_adjusted_p:.3g}")
